# 1. Wprowadzenie
- **Autor:** Karolina Stefańska s227822
- **Temat projektu:** Analiza i identyfikacja kluczowych czynników wpływających na wartość
nieruchomości w celu predykcji przyszłej wartości (ceny za noc) obiektu w regionie Azji i Pacyfiku
dostępnego do wynajmu na platformie Airbnb.
- **Cel projektu:** Głównym celem projektu jest stworzenie modelu uczenia maszynowego (regresji),
który pozwoli na przewidywanie przychodu z wynajmu Airbnb w regionie Azji i Pacyfiku. Projekt ma
na celu pomoc w podejmowaniu decyzji inwestycyjnych dla przyszłych hostów (właścicieli
apartamentów na platformie Airbnb). <br>
Projekt zakłada wykorzystanie zmiennych subiektywnych (oceny, liczba opinii) jako dodatkowych
parametrów, pozwalających na długoterminową analizę. Dzięki temu użytkownik (inwestor) będzie
mógł oszacować zarówno bazową cenę nowej oferty, jak i potencjalny wzrost przychodów po
uzyskaniu statusu Superhosta lub wysokich ocen. Dzięki temu model mógłby być również przydatny
dla obecnych właścicieli, by zweryfikować i zoptymalizować cenę za noc ich nieruchomości.

# 2. Zbiór danych
- **Źródło:** https://www.kaggle.com/datasets/jasonairroi/airbnb-market-data-asia-pacific
- **Liczba obserwacji:** 29,440
- **Liczba zmiennych:** 61, w tym finalnie 12, które zostaną wzięte pod uwagę w modelu. Ostatnie 2 zmienne z tabeli to tworzą zmienną objaśnianą
- **Cecha do przewidzenia przez model:** przychód (jako ostatnie dwie kolumny w tabeli)

In [243]:
import pandas as pd
import numpy as np
zmienne= pd.DataFrame([
    ('guests','maksymakna liczba gości', 'skokowa'),
    ('bedrooms', 'liczba sypialni', 'skokowa'),
    ('beds' ,'liczba łóżek', 'skokowa'),
    ('room_type','standard prywatności','kategoryczna (3)'),
    ('listing_type', 'standard akomodacji', 'kategoryczna') ,
    ('latitude', 'szerokość geograficza', 'ciągła'),
    ('longitude', 'długość geograficzna' ,'ciągła'),
    ('city', 'miasto', 'kategoryczna'),
    ('superhost', 'status superhosta', 'binarna'),
    ('num_reviews', 'lb. opinii', 'skokowa'),
    ('rating_overall', 'średnie opinie', 'ciągła'),
    ('photos_count', 'liczba zdjęć', 'skokowa'),
    ('instant_book', 'możliwość natychmiastowej rezerwacji', 'binarna'),
    ('l90d_revenue', 'przychód', 'ciągła'),
    ('currency', 'waluta', 'ciągła'),
    ], columns=('nazwa zmiennej', 'opis zmiennej', 'typ zmiennej'))
zmienne


,nazwa zmiennej,opis zmiennej,typ zmiennej
0,guests,maksymakna liczba gości,skokowa
1,bedrooms,liczba sypialni,skokowa
2,beds,liczba łóżek,skokowa
3,room_type,standard prywatności,kategoryczna (3)
4,listing_type,standard akomodacji,kategoryczna
5,latitude,szerokość geograficza,ciągła
6,longitude,długość geograficzna,ciągła
7,city,miasto,kategoryczna
8,superhost,status superhosta,binarna
9,num_reviews,lb. opinii,skokowa


## Wczytywanie danych i opis danych
oczyszczam dane z " na początku i końcu linii i zamieniam podwójne cudzysłowy z kolumny o udogodnieniach. Nastpęnie wczytuję dane z pliku csv pomijając wersy z błędami (wersy te nie mają listing_id)

In [244]:
with open('listings.csv', 'r', encoding='utf-8') as file:
    clean_lines=[line.strip('"').replace('""', '"') for line in file]

with open('listings_cleaned.csv', 'w', encoding='utf-8') as file_out:
    for line in clean_lines:
        file_out.write(line)

In [245]:
pd.set_option('display.max_columns', None)
listings_all_data = pd.read_csv(
    'listings_cleaned.csv', 
    quotechar='"', 
    doublequote=True, 
    skipinitialspace=True,
    on_bad_lines='skip'
)
listings_all_data.head(8)
listings_all_data.columns


C:\Users\karol\AppData\Local\Temp\ipykernel_28472\3960223325.py:2: DtypeWarning: Columns (0: photos_count, 1: latitude, 2: guests, 3: bedrooms, 4: num_reviews, 5: l90d_available_days, 6: l90d_total_days) have mixed types. Specify dtype option on import or set low_memory=False.
  listings_all_data = pd.read_csv(


Index(['listing_id', 'listing_type', 'room_type', 'cover_photo_url',
       'photos_count', 'host_id', 'superhost', 'latitude', 'longitude',
       'guests', 'bedrooms', 'beds', 'baths', 'registration', 'amenities',
       'instant_book', 'professional_management', 'min_nights',
       'cancellation_policy', 'currency', 'cleaning_fee', 'extra_guest_fee',
       'num_reviews', 'rating_overall', 'rating_accuracy', 'rating_checkin',
       'rating_cleanliness', 'rating_communication', 'rating_location',
       'rating_value', 'ttm_revenue', 'ttm_revenue_native', 'ttm_avg_rate',
       'ttm_avg_rate_native', 'ttm_occupancy', 'ttm_adjusted_occupancy',
       'ttm_revpar', 'ttm_revpar_native', 'ttm_adjusted_revpar',
       'ttm_adjusted_revpar_native', 'ttm_reserved_days', 'ttm_blocked_days',
       'ttm_available_days', 'ttm_total_days', 'l90d_revenue',
       'l90d_revenue_native', 'l90d_avg_rate', 'l90d_avg_rate_native',
       'l90d_occupancy', 'l90d_adjusted_occupancy', 'l90d_revpar',
 

tworzę ramkę danych tylko z kolumn, które mnie interesują

In [246]:
nazwy_zmiennych=list(zmienne.loc[:,'nazwa zmiennej'])
listings_df=listings_all_data.loc[:,nazwy_zmiennych]
listings_df.head(5)

,guests,bedrooms,beds,room_type,listing_type,latitude,longitude,city,superhost,num_reviews,rating_overall,photos_count,instant_book,l90d_revenue,currency
0,14,7,10.0,entire_home,Entire villa,-8.6867,115.1564,"Seminyak""",false,73.0,4.52,43.0,NaN,11007.0,IDR
1,6,3,3.0,entire_home,Entire villa,-8.6958,115.1725,"Seminyak""",true,152.0,4.72,21.0,NaN,5759.0,IDR
2,10,4,4.0,entire_home,Entire villa,-8.6927,115.1657,"Seminyak""",true,118.0,4.71,18.0,NaN,5110.0,IDR
3,8,4,5.0,entire_home,Entire villa,-8.6905,115.1640,"Seminyak""",true,127.0,4.91,80.0,NaN,22294.0,IDR
4,14,7,8.0,entire_home,Entire villa,-8.6908,115.1632,"Seminyak""",true,6.0,4.80,63.0,NaN,16164.0,IDR


In [247]:
listings_df['city'] = listings_df['city'].map(lambda x: str(x).strip('"'))
listings_df.head(3)

,guests,bedrooms,beds,room_type,listing_type,latitude,longitude,city,superhost,num_reviews,rating_overall,photos_count,instant_book,l90d_revenue,currency
0,14,7,10.0,entire_home,Entire villa,-8.6867,115.1564,Seminyak,false,73.0,4.52,43.0,NaN,11007.0,IDR
1,6,3,3.0,entire_home,Entire villa,-8.6958,115.1725,Seminyak,true,152.0,4.72,21.0,NaN,5759.0,IDR
2,10,4,4.0,entire_home,Entire villa,-8.6927,115.1657,Seminyak,true,118.0,4.71,18.0,NaN,5110.0,IDR


In [248]:
listings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29169 entries, 0 to 29168
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   guests          24614 non-null  object 
 1   bedrooms        22476 non-null  object 
 2   beds            28292 non-null  float64
 3   room_type       28787 non-null  str    
 4   listing_type    28787 non-null  str    
 5   latitude        28786 non-null  object 
 6   longitude       28786 non-null  float64
 7   city            29169 non-null  str    
 8   superhost       28785 non-null  str    
 9   num_reviews     28748 non-null  object 
 10  rating_overall  27832 non-null  float64
 11  photos_count    28497 non-null  object 
 12  instant_book    3412 non-null   str    
 13  l90d_revenue    28779 non-null  float64
 14  currency        28784 non-null  str    
dtypes: float64(4), object(5), str(6)
memory usage: 3.3+ MB


In [249]:
listings_df['guests'] = pd.to_numeric(listings_df['guests'], errors='coerce')
listings_df['bedrooms'] = pd.to_numeric(listings_df['bedrooms'], errors='coerce')
listings_df['latitude'] = pd.to_numeric(listings_df['latitude'], errors='coerce')
listings_df['num_reviews'] = pd.to_numeric(listings_df['num_reviews'], errors='coerce')
listings_df['photos_count'] = pd.to_numeric(listings_df['photos_count'], errors='coerce')
listings_df['superhost'] = listings_df['superhost'].map(lambda x: False if x=='False' else True)

In [250]:
listings_df.describe()

,guests,bedrooms,beds,latitude,longitude,num_reviews,rating_overall,photos_count,l90d_revenue
count,24613.000000,22475.000000,28292.000000,28785.000000,28786.000000,28747.000000,27832.000000,2.849600e+04,2.877900e+04
mean,4.796774,2.060156,2.646861,5.603658,119.627171,121.817380,4.761141,7.439223e+04,3.282858e+03
std,3.272582,1.544837,2.356941,22.970648,23.574772,325.902086,0.317681,4.458833e+06,1.523480e+04
min,1.000000,0.000000,0.000000,-45.076700,-121.981700,0.000000,2.500000,0.000000e+00,0.000000e+00
25%,2.000000,1.000000,1.000000,-8.513800,102.262200,28.000000,4.670000,1.700000e+01,1.500000e+02
50%,4.000000,2.000000,2.000000,10.364100,120.300900,74.000000,4.810000,2.700000e+01,9.520000e+02
75%,6.000000,3.000000,3.000000,19.070200,128.946625,165.000000,4.900000,4.100000e+01,3.208000e+03
max,16.000000,41.000000,42.000000,129.274500,176.309400,50358.000000,45.700000,3.740323e+08,2.316489e+06


## Analiza brakujących wartości
Sprawdzam procentową zawartość braków danych w każdej kolumnie. Najwięcej ich jest w instant_book, więc w następnym kroku pozbędę się tej kolumny. Dużo braków danych jest również w guests oraz bedrooms. 

In [251]:
listings_df.isnull().sum()/len(listings_df)

guests            0.156193
bedrooms          0.229490
beds              0.030066
room_type         0.013096
listing_type      0.013096
latitude          0.013165
longitude         0.013130
city              0.000000
superhost         0.000000
num_reviews       0.014467
rating_overall    0.045836
photos_count      0.023072
instant_book      0.883027
l90d_revenue      0.013370
currency          0.013199
dtype: float64

In [252]:
listings_df=listings_df.drop('instant_book', axis=1)

In [253]:
listings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29169 entries, 0 to 29168
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   guests          24613 non-null  float64
 1   bedrooms        22475 non-null  float64
 2   beds            28292 non-null  float64
 3   room_type       28787 non-null  str    
 4   listing_type    28787 non-null  str    
 5   latitude        28785 non-null  float64
 6   longitude       28786 non-null  float64
 7   city            29169 non-null  str    
 8   superhost       29169 non-null  bool   
 9   num_reviews     28747 non-null  float64
 10  rating_overall  27832 non-null  float64
 11  photos_count    28496 non-null  float64
 12  l90d_revenue    28779 non-null  float64
 13  currency        28784 non-null  str    
dtypes: bool(1), float64(9), str(4)
memory usage: 2.9 MB


Zajmuję się usunięciem wartości nan z pierwszych trzech kolumn

In [254]:
listings_df[['guests', 'bedrooms', 'beds']].info()

<class 'pandas.DataFrame'>
RangeIndex: 29169 entries, 0 to 29168
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   guests    24613 non-null  float64
 1   bedrooms  22475 non-null  float64
 2   beds      28292 non-null  float64
dtypes: float64(3)
memory usage: 683.8 KB


Obliczam ile łóżek przypada średnio na każdą sypialnię by uzupełnić te wartości i zaokrąglę wynik do pełnej liczby

In [255]:
dzielenie=listings_df['bedrooms']/listings_df['beds']
mask0=(dzielenie>0) & (dzielenie.notnull()) & (dzielenie!=np.inf)
dzielenie_rooms_beds=dzielenie[mask0].mean()
print(dzielenie_rooms_beds)

0.8161491197254633


In [256]:
dzielenie=listings_df['guests']/listings_df['bedrooms']
dzielenie_guests_rooms=dzielenie[mask0].mean()
print(dzielenie_guests_rooms)

2.5965980683025522


In [257]:
mask1 = listings_df['bedrooms'].isnull()
listings_df.loc[mask1, 'bedrooms'] = listings_df.loc[mask1, 'beds'] *dzielenie_rooms_beds

mask1 = listings_df['bedrooms'].isnull() #aktualizacja maski
listings_df.loc[mask1, 'bedrooms'] = listings_df.loc[mask1, 'guests'] / dzielenie_guests_rooms

mask2 = listings_df['guests'].isnull()
listings_df.loc[mask2, 'guests'] = listings_df.loc[mask2, 'bedrooms'] * dzielenie_guests_rooms

mask3 = listings_df['beds'].isnull()
listings_df.loc[mask3, 'beds'] = listings_df.loc[mask3, 'bedrooms'] / dzielenie_rooms_beds

In [258]:
listings_df = listings_df.dropna(subset=['guests', 'bedrooms', 'beds'])

Po usunięciu braków danych z pierwszych trzech kolumn nie ma wiele więcej wierszy z wartościami pustymi, w kolumnach które mają pojedyncze braki danych, te rekordy po prostu opuszczam

In [259]:
listings_df.info()

<class 'pandas.DataFrame'>
Index: 28768 entries, 0 to 29168
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   guests          28768 non-null  float64
 1   bedrooms        28768 non-null  float64
 2   beds            28768 non-null  float64
 3   room_type       28768 non-null  str    
 4   listing_type    28768 non-null  str    
 5   latitude        28767 non-null  float64
 6   longitude       28768 non-null  float64
 7   city            28768 non-null  str    
 8   superhost       28768 non-null  bool   
 9   num_reviews     28729 non-null  float64
 10  rating_overall  27814 non-null  float64
 11  photos_count    28477 non-null  float64
 12  l90d_revenue    28761 non-null  float64
 13  currency        28766 non-null  str    
dtypes: bool(1), float64(9), str(4)
memory usage: 3.1 MB


In [260]:
listings_df = listings_df.dropna(subset=['latitude', 'superhost', 'l90d_revenue', 'city'])

columns_to_clean=['num_reviews', 'rating_overall', 'photos_count']
listings_df[columns_to_clean]=listings_df[columns_to_clean].fillna(listings_df[columns_to_clean].mean())

In [261]:
listings_df.to_csv('listings_used.csv', index=False)

## Analiza wartości odstających

In [262]:
listings_df.describe()

,guests,bedrooms,beds,latitude,longitude,num_reviews,rating_overall,photos_count,l90d_revenue
count,28760.000000,28760.000000,28760.000000,28760.000000,28760.000000,28760.000000,28760.000000,2.876000e+04,2.876000e+04
mean,4.632161,1.886684,2.635292,5.557711,119.649248,119.978043,4.759615,4.286399e+04,3.283113e+03
std,3.337199,1.478726,2.369003,22.900002,23.502522,135.559608,0.198416,3.453414e+06,1.523974e+04
min,0.000000,0.000000,0.000000,-45.076700,-121.981700,0.000000,2.500000,0.000000e+00,0.000000e+00
25%,2.000000,1.000000,1.000000,-8.514475,102.262275,28.000000,4.680000,1.700000e+01,1.490000e+02
50%,4.000000,1.000000,2.000000,10.354700,120.300300,74.000000,4.800000,2.700000e+01,9.515000e+02
75%,6.000000,2.000000,3.000000,19.063125,128.939600,164.250000,4.890000,4.200000e+01,3.205000e+03
max,72.704746,41.000000,50.235918,129.274500,176.309400,1523.000000,5.000000,3.387967e+08,2.316489e+06


- Już na wstępie widać, że mamy oferty z zerowym przychodem oraz takie, które przez ostatnie 90 dni zarobiły ponad 2 mln, te wartości odstające należy usunąć. 
    - W pierwszym kroku usunę wszystkie wartości zerowe
    - Następnie zawężę analizowane dane, aby nie uwzględniały najbardziej odstających ofert
- Do tego nie możliwy jest photos_count na tak wysokim poziomie, jednak 75% ofert ma jedynie około 43 zdjęcia, dlatego należy zmienić wartości najbardziej odstające
    - W tym celu zrównam skrajnie duże wartości 
- Należałoby również sprawdzić za pomocą wykresu punktowego czy nie ma błędów w wartościach współrzędnych geograficznych

### Usunięcie lokazji ze skrajnymi przychodami

In [263]:
listings_df=listings_df[listings_df['l90d_revenue']>0]

In [264]:
Q1_revenue=listings_df['l90d_revenue'].quantile(0.25)
Q3_revenue=listings_df['l90d_revenue'].quantile(0.75)

IQR_revenue = Q3_revenue - Q1_revenue

lower_revenue=Q1_revenue-1.5*IQR_revenue
upper_revenue=Q3_revenue+1.5*IQR_revenue
df_cleaned = listings_df[(listings_df['l90d_revenue'] >= lower_revenue) & (listings_df['l90d_revenue'] <= upper_revenue)]
len(listings_df)-len(df_cleaned)

2393

usunie to wiele rekordów, należy przyjąć inną strategię, spróbuję usunąć jedynie rekordy poniżej percentyla 0.05 oraz powyżej 0.95

In [265]:
lower_revenue=listings_df['l90d_revenue'].quantile(0.05)
upper_revenue=listings_df['l90d_revenue'].quantile(0.95)
df_cleaned=listings_df[(listings_df['l90d_revenue'] >= lower_revenue) & (listings_df['l90d_revenue'] <= upper_revenue)]
len(listings_df)-len(df_cleaned)
print(len(df_cleaned))

20923


In [266]:
listings_df=df_cleaned
del df_cleaned

### Wyrównanie liczby zdjęć

Sprawdzam, jaka jest największa sensowna wartość photos_count, która nie zaburzy wnioskowania modelu: jest to percentyl 0.98

In [273]:
max_photos=listings_df['photos_count'].quantile(0.98)
print(max_photos)

118.0


In [288]:
mask=listings_df['photos_count']>max_photos
listings_df.loc[mask,'photos_count']=max_photos

In [289]:
listings_df.describe()

,guests,bedrooms,beds,latitude,longitude,num_reviews,rating_overall,photos_count,l90d_revenue
count,20923.000000,20923.000000,20923.000000,20923.000000,20923.000000,20923.000000,20923.000000,20923.000000,20923.000000
mean,6.849101,4.152088,4.872629,5.821304,119.643000,131.595057,4.768199,33.615160,2839.171247
std,16.190886,16.307832,16.293170,22.793915,23.307854,142.521363,0.186942,23.057692,3272.860171
min,0.000000,0.000000,0.000000,-45.076700,2.000000,1.000000,2.500000,1.000000,111.000000
25%,2.000000,1.000000,1.000000,-8.501500,102.251050,33.000000,4.690000,17.000000,595.000000
50%,4.000000,1.000000,2.000000,10.706700,120.330200,85.000000,4.810000,27.000000,1496.000000
75%,6.000000,2.000000,3.000000,19.140750,128.945400,181.000000,4.900000,43.000000,3744.000000
max,118.000000,118.000000,118.000000,101.611200,176.309400,1523.000000,5.000000,118.000000,15647.000000


In [297]:
print((listings_df['listing_type'].value_counts(normalize=True) * 100).tail(20))

listing_type
Shared room in guest suite            0.009559
Shipping container                    0.004779
Shared room in cottage                0.004779
Holiday park                          0.004779
Entire home/apt                       0.004779
Private room in lighthouse            0.004779
Private room in camper/rv             0.004779
Shared room in villa                  0.004779
Entire hostel                         0.004779
Private room in religious building    0.004779
entire_home                           0.004779
cafe street                           0.004779
Shared room in townhouse              0.004779
Shared room in dome                   0.004779
Shared room in apartment              0.004779
Entire resort                         0.004779
Boat                                  0.004779
Shared room                           0.004779
Private room in floor                 0.004779
Tower                                 0.004779
Name: proportion, dtype: float64


## Statystyki

In [ ]:
listings_df.describe()

,guests,bedrooms,beds,latitude,longitude,num_reviews,rating_overall,photos_count,l90d_revenue
count,20923.000000,20923.000000,20923.000000,20923.000000,20923.000000,20923.000000,20923.000000,2.092300e+04,20923.000000
mean,4.590120,1.834829,2.578147,5.821304,119.643000,131.595057,4.768199,1.669225e+04,2839.171247
std,3.259910,1.337809,2.195998,22.793915,23.307854,142.521363,0.186942,2.342218e+06,3272.860171
min,0.000000,0.000000,0.000000,-45.076700,2.000000,1.000000,2.500000,1.000000e+00,111.000000
25%,2.000000,1.000000,1.000000,-8.501500,102.251050,33.000000,4.690000,1.700000e+01,595.000000
50%,4.000000,1.000000,2.000000,10.706700,120.330200,85.000000,4.810000,2.700000e+01,1496.000000
75%,6.000000,2.000000,3.000000,19.140750,128.945400,181.000000,4.900000,4.300000e+01,3744.000000
max,72.704746,41.000000,50.235918,101.611200,176.309400,1523.000000,5.000000,3.387967e+08,15647.000000


In [ ]:
print(listings_df['room_type'].value_counts(normalize=True) * 100)

room_type
entire_home                                                                     75.648808
private_room                                                                    22.377288
hotel_room                                                                       1.237872
shared_room                                                                      0.726473
https://a0.muscache.com/im/pictures/8e861dc2-161b-4e73-8f93-7afe1a296c90.jpg     0.004779
close to the air\",Pension,entire_home,a5aeb87476a8,207121254                    0.004779
Name: proportion, dtype: float64


In [296]:
print((listings_df['listing_type'].value_counts(normalize=True) * 100).tail(20))

listing_type
Shared room in guest suite            0.009559
Shipping container                    0.004779
Shared room in cottage                0.004779
Holiday park                          0.004779
Entire home/apt                       0.004779
Private room in lighthouse            0.004779
Private room in camper/rv             0.004779
Shared room in villa                  0.004779
Entire hostel                         0.004779
Private room in religious building    0.004779
entire_home                           0.004779
cafe street                           0.004779
Shared room in townhouse              0.004779
Shared room in dome                   0.004779
Shared room in apartment              0.004779
Entire resort                         0.004779
Boat                                  0.004779
Shared room                           0.004779
Private room in floor                 0.004779
Tower                                 0.004779
Name: proportion, dtype: float64


In [ ]:
print(listings_df['city'].value_counts(normalize=True) * 100)

city
Osaka           1.280887
Baguio          1.247431
Tagaytay        1.228313
George Town     1.175740
Christchurch    1.156622
                  ...   
Noosa Heads     0.525737
Vũng Tàu        0.525737
Byron Bay       0.497061
Gurugram        0.463605
nan             0.004779
Name: proportion, Length: 113, dtype: float64


Wioski:
- należy usunąc wiersze z odstającymi typami pokojów
- należy usunąć dane z najbardziej skrajnymi przychodami 
- problemem jest kolumna z licznikiem zdjęć z wartościami niemożliwymi, należy je zmniejszyć
- należy również sprawdzić za pomocą wykresu punktowego czy nie ma błędów w wartościach współrzędnych geograficznych
- należy dodać kolumnę oznaczającą odległość od centrum miasta by mieć bardziej miarodajne informacje do analizy dla modelu